# Tarea 1 – Analítica de Datos (MACI 2026)

**Alumno:** Keyden Urbina

**Mercado de celulares en Chile (solotodo.cl) y modelo de rango de precios**

## Notebook 01 · Web scraping → `data/output/data_raw.csv`

Extrae **solo desde el HTML** de solotodo.cl (requests + BeautifulSoup, **sin la API pública**) las características, características avanzadas, accesorios y los precios de cada celular en todas las tiendas, y si es reacondicionado o no. Guarda los datos **sin limpiar**.

| Indicación del curso | Cómo se cumple |
|---|---|
| 0. Guardar lo descargado a medida que avanzas | Cada página se guarda comprimida en `data/input/html_AAAAMMDD/` apenas se descarga. Si el proceso se corta, al re-ejecutar se lee del disco y solo se baja lo que falta. |
| 1. Paginación `?page=N` y `time.sleep(1)` | El listado se recorre con `/celulares?page=1, 2, 3…` hasta que una página no trae celulares nuevos. Hay 1 s de pausa entre requests. |
| 2. Páginas de modelo (`/iphone-17`) | Se lee el `<script>` con `"products":[{"id":…}]` y se usa la **primera variante** de la lista. |
| 3. Precios ocultos ("Ver más precios") | Las ofertas de **todas** las tiendas salen del `<script type="application/ld+json">` (tienda y condición). El precio normal, el precio oferta, el cupón y si la oferta exige plan salen de los datos de cada oferta que la página trae embebidos en sus `<script>`. Ambos están en el HTML. |
| 4. Números desde texto | Se resuelve en el notebook 02 (limpieza), con expresiones regulares. |
| 5. ≥ 100 productos de distintas páginas | Se recorre el catálogo completo: todas las páginas del listado. |
| 6. requests + BeautifulSoup, sin API | Toda la información sale del HTML de las páginas públicas del sitio. |

**Cómo ejecutarlo:** en tu computador (VS Code o Jupyter), con *Ejecutar todo*. Tarda ~10–15 minutos la primera vez (≈ 260 páginas × 1 s de pausa); si se re-ejecuta el mismo día, usa lo ya descargado. Antes de recorrer todo, el Paso 3 prueba con 1 producto.

### Paso 0 · Librerías y rutas

In [1]:
# Librerías: todas vienen con Anaconda / Colab
import re, json, time, gzip
import pandas as pd, requests
from bs4 import BeautifulSoup                         # lee HTML como un árbol de etiquetas
from requests.adapters import HTTPAdapter, Retry      # reintentos automáticos si el sitio falla
from tqdm.auto import tqdm                            # barra de progreso
from pathlib import Path

In [2]:
# Carpeta "data" del proyecto: la busca subiendo desde donde se ejecuta (sirve desde notebooks/ o desde la raíz)
DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), Path("data"))
OUT = DATA / "output"                                 # aquí queda data_raw.csv
ahora = pd.Timestamp.now(tz="America/Santiago")       # hora de Chile (Colab corre en hora UTC)
CACHE = DATA / "input" / f"html_{ahora:%Y%m%d}"       # HTML descargado hoy: si el proceso se corta, se retoma desde aquí
for carpeta in (OUT, CACHE): carpeta.mkdir(parents=True, exist_ok=True)
marca = CACHE / "inicio.txt"                          # anota la hora en que empezó la descarga de hoy (solo la 1ª vez)
if not marca.exists(): marca.write_text(ahora.isoformat())
inicio = pd.Timestamp(marca.read_text())              # al re-ejecutar se usa la hora de la descarga original, no la de ahora
print("RAW →", OUT.resolve(), "\nHTML →", CACHE.resolve(), "\nDescarga iniciada:", f"{inicio:%Y-%m-%d %H:%M}")

RAW → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/output 
HTML → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/input/html_20261107


### Paso 1 · Descarga con respaldo en disco
`bajar()` es la única función que habla con el sitio: descarga una página, la guarda comprimida apenas llega y espera 1 s. Si esa página ya se bajó hoy, la lee del disco y no vuelve a pedirla.

In [3]:
BASE = "https://www.solotodo.cl"
s = requests.Session()                                # reutiliza la conexión: más rápido que abrir una por página
s.headers["User-Agent"] = "Mozilla/5.0"               # nos presentamos como un navegador común
s.mount("https://", HTTPAdapter(max_retries=Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))

def bajar(ruta, nombre):
    """HTML de BASE + ruta. Si ya se bajó hoy, lo lee del disco; si no, lo descarga, lo guarda comprimido y espera 1 s."""
    archivo = CACHE / f"{nombre}.html.gz"
    if archivo.exists():                              # ya descargado → no se vuelve a pedir al sitio
        return gzip.open(archivo, "rt", encoding="utf-8").read()
    r = s.get(BASE + ruta, timeout=30)
    r.raise_for_status()                              # si el sitio responde error (403, 404…), se detiene en vez de guardar basura
    with gzip.open(archivo, "wt", encoding="utf-8") as f: f.write(r.text)   # se guarda apenas se descarga
    time.sleep(1)                                     # pausa de 1 s entre requests, para no sobrecargar el sitio
    return r.text

### Paso 2 · Listado de celulares (paginación)

In [4]:
# Paso 1 · Listado: recorre /celulares?page=1, 2, 3… y junta los enlaces de cada tarjeta (20 por página)
enlaces, pag = [], 1
while True:
    sopa = BeautifulSoup(bajar(f"/celulares?page={pag}", f"listado_{pag:02d}"), "html.parser")
    hrefs = [a["href"] for a in sopa.find_all("a", href=True)]
    nuevos = [h for h in dict.fromkeys(hrefs)                                  # dict.fromkeys: quita repetidos y mantiene el orden
              if (h.startswith("/products/") or "?product=" in h) and h not in enlaces]   # tarjetas: producto o página de modelo
    if not nuevos: break                                                       # página sin celulares nuevos → se acabó el listado
    enlaces += nuevos
    pag += 1
modelos = [h for h in enlaces if not h.startswith("/products/")]
print(f"{len(enlaces)} celulares en {pag - 1} páginas | {len(modelos)} enlaces a página de modelo (ej: {modelos[:2]})")

241 celulares en 13 páginas | 13 enlaces a página de modelo (ej: ['/iphone-17?product=355682', '/iphone-17-pro-max?product=355715'])


### Paso 3 · De cada enlace a su ficha, con todas sus ofertas
Se prueba primero con **1 producto**, como recomienda la tarea. La tabla debe mostrar varias tiendas, con precio normal y precio oferta.

In [5]:
# Paso 2 · Enlace → id del producto. Las páginas de modelo (/iphone-17) no muestran precios:
# su HTML trae en un <script> la lista "products":[{"id":…}] y se usa la PRIMERA variante (instrucción de la tarea).
def id_producto(href):
    if href.startswith("/products/"):
        return re.search(r"/products/(\d+)", href)[1]
    ruta = href.split("?")[0]                                                  # "/iphone-17?product=355682" → "/iphone-17"
    html = bajar(ruta, "modelo_" + ruta.strip("/")).replace('\\"', '"')        # el <script> trae las comillas escapadas: \" → "
    return re.search(r'"products":\[\{"id":(\d+)', html)[1]

# Paso 3 · Ficha del producto (todo sale del HTML):
#  a) características, avanzadas y accesorios → pares <dt>etiqueta</dt><dd>valor</dd>
#  b) ofertas de TODAS las tiendas (no solo las 5 visibles) → <script type="application/ld+json">: tienda, URL y condición
#  c) precio normal, precio oferta, cupón y si la oferta exige plan → datos de cada oferta que la página trae embebidos
#     en sus <script> (self.__next_f.push). Se cruzan con (b) por la URL de la tienda.
def producto(pid):
    html = bajar(f"/products/{pid}", f"producto_{pid}")
    sopa = BeautifulSoup(html, "html.parser")
    tag = sopa.find("script", type="application/ld+json")
    ld = json.loads(tag.string) if tag else {}                                 # sin ofertas, la página no trae ld+json
    marca = ld.get("brand", {})
    base = {"nombre": ld.get("name") or sopa.h1.get_text(strip=True),
            "marca": marca.get("name") if isinstance(marca, dict) else marca,
            "id": pid,
            **{dt.get_text(strip=True): " ".join(dd.get_text().split())        # " ".join(...split()) deja el texto en una sola línea
               for dt, dd in zip(sopa.find_all("dt"), sopa.find_all("dd"))}}
    tienda = {o["url"]: o["seller"]["name"] for o in ld.get("offers", {}).get("offers", [])}   # URL de la oferta → nombre de la tienda

    # (c) Los <script> self.__next_f.push([1,"…"]) guardan texto escapado: json.loads lo desescapa y se unen en uno solo
    texto = "".join(json.loads(f'"{t}"') for t in re.findall(r'self\.__next_f\.push\(\[1,"((?:[^"\\]|\\.)*)"\]\)', html))
    i = texto.find('"entities":')
    ofertas = json.JSONDecoder().raw_decode(texto, i + len('"entities":'))[0] if i >= 0 else []   # lee solo la lista [...]

    filas = [{**base, "tienda": tienda.get(e["external_url"]),
              "precio_normal": e["active_registry"]["normal_price"],
              "precio_oferta": e["active_registry"]["offer_price"],
              "condicion": e["condition"],                                     # NewCondition / RefurbishedCondition / UsedCondition
              "cupon_tipo": (e["best_coupon"] or {}).get("amount_type_text"),  # cupón de descuento, si existe
              "cupon_monto": (e["best_coupon"] or {}).get("amount")}
             for e in ofertas if e["cell_plan"] is None]                       # solo equipos "Liberados" (sin plan de compañía)
    return filas or [base]                                                     # sin ofertas liberadas: queda la ficha sin precios

# Prueba con 1 producto antes de recorrer todo (recomendación de la tarea)
prueba = pd.DataFrame(producto(id_producto(enlaces[0])))
prueba[["nombre", "marca", "tienda", "precio_normal", "precio_oferta", "condicion", "cupon_tipo"]]

,nombre,marca,tienda,precio_normal,precio_oferta,condicion,cupon_tipo
0,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Movistar,549990.00,549990.00,https://schema.org/RefurbishedCondition,None
1,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Reuse,739990.00,739990.00,https://schema.org/RefurbishedCondition,Percentage
2,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Digitek,759990.00,759990.00,https://schema.org/UsedCondition,None
3,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Lider,799990.00,799990.00,https://schema.org/NewCondition,None
4,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Movistar,879990.00,879990.00,https://schema.org/NewCondition,None
5,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Falabella,949990.00,899990.00,https://schema.org/NewCondition,None
6,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Entel,1469990.00,999990.00,https://schema.org/NewCondition,None
7,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,ABC,1099990.00,1099990.00,https://schema.org/NewCondition,None
8,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Ripley,1179990.00,1159990.00,https://schema.org/NewCondition,None
9,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Samsung Shop,1469990.00,1469990.00,https://schema.org/NewCondition,None


#### Prueba con 2 páginas (~40 celulares)
Antes de recorrer todo, se procesan los celulares de las 2 primeras páginas del listado (recomendación de la tarea). Si algo falla, falla aquí y no a mitad de las 13 páginas. Lo descargado queda guardado, así que el Paso 4 no lo vuelve a pedir.

In [ ]:
prueba2 = pd.DataFrame([f for h in enlaces[:40] for f in producto(id_producto(h))])   # ~2 páginas × 20 celulares
print(prueba2.shape, "| celulares:", prueba2.nombre.nunique(), "| tiendas:", prueba2.tienda.nunique(),
      "| sin precio liberado:", prueba2.precio_oferta.isna().sum(),
      "| ofertas sin nombre de tienda:", (prueba2.precio_oferta.notna() & prueba2.tienda.isna()).sum())   # debe ser 0

### Paso 4 · Todos los celulares → `data_raw.csv`

In [6]:
# Paso 4 · Recorre todos los celulares y guarda data_raw.csv (1 fila = 1 celular en 1 tienda)
filas = []
for h in tqdm(enlaces):
    try:
        filas += producto(id_producto(h))
    except Exception as e:                                                     # un celular con problemas no detiene todo
        print("Error en", h, "→", e)

raw = pd.DataFrame(filas)
raw["fecha_extraccion"] = inicio.strftime("%Y-%m-%d %H:%M")                    # hora en que se descargaron los precios
archivo = f"data_raw_{inicio:%Y%m%d_%H%M}.csv"                                 # ej: data_raw_20261107_1214.csv (re-ejecutar no crea otro)
raw.to_csv(OUT / archivo, index=False)                                         # copia histórica: cada ejecución crea un archivo nuevo
raw.to_csv(OUT / "data_raw.csv", index=False)                                  # versión vigente, con el nombre que pide el enunciado
print(raw.shape, "| celulares:", raw.nombre.nunique(), "→", OUT / archivo)
raw.head()

  0%|          | 0/241 [00:00<?, ?it/s]

(958, 32) | celulares: 241 → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/output/data_raw_20261107_1254.csv


,nombre,marca,id,Pantalla,Memoria interna,RAM,Cámara principal,Cámara frontal,Batería,Sistema operativo,...,NFC,Bluetooth,Lector de huellas digitales,tienda,precio_normal,precio_oferta,condicion,cupon_tipo,cupon_monto,fecha_extraccion
0,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Sí,Sí,Sí,Tienda Movistar,549990.00,549990.00,https://schema.org/RefurbishedCondition,None,None,2026-11-07 12:54
1,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Sí,Sí,Sí,Reuse,739990.00,739990.00,https://schema.org/RefurbishedCondition,Percentage,5.00,2026-11-07 12:54
2,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Sí,Sí,Sí,Digitek,759990.00,759990.00,https://schema.org/UsedCondition,None,None,2026-11-07 12:54
3,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Sí,Sí,Sí,Lider,799990.00,799990.00,https://schema.org/NewCondition,None,None,2026-11-07 12:54
4,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Sí,Sí,Sí,Tienda Movistar,879990.00,879990.00,https://schema.org/NewCondition,None,None,2026-11-07 12:54


### Decisiones de la extracción (justificadas)
- **Variante en las páginas de modelo:** se usa la **primera de la lista**, como indica la tarea. Puede no ser la misma que muestra la tarjeta del listado: el 07-11, en 9 de las 13 páginas de modelo la primera variante era otra capacidad o color.
- **Ofertas con plan de compañía:** se excluyen. Su precio es el de un arriendo con plan (ej. \$9.990 + cuota mensual), no el del equipo, y la ficha las muestra aparte de los equipos "Liberados".
- **Celulares sin ofertas liberadas:** se conservan con su ficha y sin precio. La limpieza (notebook 02) decide qué hacer con ellos.
- **Cupones:** se guardan tal cual (tipo y monto). El descuento se aplica en la limpieza, porque el enunciado considera el precio con cupón como precio oferta.
- **Duplicados y reacondicionados:** se guardan todos sin filtrar (el RAW va sin limpiar). Se tratan en el notebook 02.

**Resultado esperado** (simulación sobre el sitio, 07-11-2026): 13 páginas, **244 celulares** (13 vía página de modelo), ≈ 977 ofertas liberadas, 72 ofertas con plan excluidas, 10 celulares sin oferta liberada y 54 ofertas con cupón.